# Batch Integration

Low-level 1-D and 2-D integration reference. The smoke path makes a
compact synthetic pattern; real mode uses an explicit image and PONI.
For a durable multi-frame reduction, continue to notebook 06.


In [ ]:
import os
from IPython import get_ipython

# Equivalent to %matplotlib widget; headless checks explicitly use inline.
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", os.environ.get("XDART_NOTEBOOK_BACKEND", "widget"))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output, display

from xrd_tools.core.containers import IntegrationResult1D, IntegrationResult2D
from xrd_tools.integrate import integrate_1d, integrate_2d, load_poni
from xrd_tools.io import read_image
from xrd_tools.viz import plot_1d


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
image_file = TEST_DATA / "image.tif"
poni_file = TEST_DATA / "calibration.poni"
npt_1d = widgets.BoundedIntText(value=256, min=32, max=4096, description="1-D points")
npt_azim = widgets.BoundedIntText(value=72, min=8, max=720, description="chi points")
run_button = widgets.Button(description="Run integration", button_style="primary")
status = widgets.HTML("<i>Configure real paths, then run explicitly.</i>")
output = widgets.Output()
display(widgets.VBox([widgets.HBox([npt_1d, npt_azim]), run_button, status, output]))


In [ ]:
NOTEBOOK_STATE = {"runs": 0, "result_1d": None, "result_2d": None}

def run_integration(_=None):
    # Run both public integration operations with current widget values.
    with output:
        clear_output(wait=True)
        try:
            if SMOKE_MODE:
                q = np.linspace(1.0, 4.0, npt_1d.value)
                chi = np.linspace(-35.0, 35.0, npt_azim.value)
                intensity = 20 + 90 * np.exp(-0.5 * ((q - 2.35) / 0.07) ** 2)
                cake = np.outer(intensity, 1.0 + 0.1 * np.cos(np.deg2rad(chi)))
                result_1d = IntegrationResult1D(q, intensity, unit="q_A^-1")
                result_2d = IntegrationResult2D(q, chi, cake, unit="q_A^-1", azimuthal_unit="chi_deg")
            else:
                assert image_file.is_file(), f"Missing detector image: {image_file}"
                assert poni_file.is_file(), f"Missing PONI calibration: {poni_file}"
                image, poni = read_image(image_file), load_poni(poni_file)
                result_1d = integrate_1d(image, poni, npt=npt_1d.value)
                result_2d = integrate_2d(image, poni, npt_rad=npt_1d.value, npt_azim=npt_azim.value)
            fig, axes = plt.subplots(1, 2, figsize=(11, 3))
            plot_1d(axes[0], result_1d.radial, result_1d.intensity, fmt="-", attrs={"xlabel": f"q ({result_1d.unit})", "ylabel": "Intensity", "title": "1-D integration"})
            axes[1].pcolormesh(result_2d.radial, result_2d.azimuthal, result_2d.intensity.T, shading="auto")
            axes[1].set(xlabel=f"q ({result_2d.unit})", ylabel=result_2d.azimuthal_unit, title="2-D integration")
            plt.show()
            NOTEBOOK_STATE.update(runs=NOTEBOOK_STATE["runs"] + 1, result_1d=result_1d, result_2d=result_2d)
            status.value = f"<b>Integrated {npt_1d.value} q points and {npt_azim.value} chi points.</b>"
        except Exception as exc:
            status.value = f"<b>Integration failed:</b> {exc}"
            raise

run_button.on_click(run_integration)
NOTEBOOK_ACTIONS = {"run_integration": run_integration}
if SMOKE_MODE or os.environ.get("XDART_NOTEBOOK_AUTORUN") == "1":
    run_integration()
